# Develop and test a QEC protocol (qodec)

Use `qdk` and `qodec` to develop and test a quantum error correction protocol. Start with the C4 code, and build a baseline qodec. Then inspect its gadgets and improve them. Along the way, use `qdk` built-in profile and audit features to debug and validate.

## The desired result (TODO)
 TODO: Show a plot or some other metrics of a working 4.2.2 protocol.

## Install

```bash
pip install "qdk[ec]"
```

## 1. Define your code

We will base our protocol around the $4$-qubit $[[4,2,2]]$ error detecting code C4.
The C4 code stores two logical qubits in four physical qubits. Its two stabilizers are $XXXX$ and $ZZZZ$. 

The `qodec` package is used to define the protocol; `qdk` provides the analysis.

In [1]:
import qodec

c4 = qodec.Code(
    "C4",
    stabilizers=["X_0 X_1 X_2 X_3", "Z_0 Z_1 Z_2 Z_3"],
    x=["X_0 X_1", "X_0 X_2"],
    z=["Z_0 Z_2", "Z_0 Z_1"],
)

Pauli strings in `qodec` are given by a sparse notation `P_i`, where `P` is a Pauli matrix `{X,Y,Z}` and `i` is the index of the qubit on which the Pauli acts.

### Check the code distance

We've defined the code. Now we'd like to make sure it's _correct_. In particular, it should have distance two.  QDK provides a `CodeProfile` that, among other things, can compute the code distance.

In [2]:
import qdk.ec as ec

code = ec.CodeProfile(c4)
code_distance = code.distance()
print("Code distance:", code_distance)

error = code_distance.witness.product
syndrome = code.syndrome_of(error)
effect = code.logical_effect_of(error)

print(f"Witness: {error}, (syndrome: {list(syndrome)}, logical effect: {effect})")
assert code_distance == error.weight == 2

Code distance: 2
Witness: XX, (syndrome: [], logical effect: X)


## 2. Build a baseline qodec

We've checked the code. Now we need circuits that prepare, measure, and operate on its logical qubits. QDK provides `build_qodec` to build a starting protocol from the code definition.

We will use `bare-css/v1`, a strategy that measures syndromes with ancillas but adds no flag qubits.

The result has two layers: logical C4 operations and physical Stim operations. Each logical instruction has a _gadget_: a circuit that implements it using the next layer. Its encodings identify the physical qubits for each logical block; its equations describe checks and readouts.

Let's build the qodec and inspect the start of its YAML bundle. We will use `gadgets` to edit the protocol directly.

In [3]:
protocol = ec.build_qodec(c4, strategy="bare-css/v1", strict=False)
logical_layer = protocol.layers[0]
gadgets = logical_layer.gadgets
print(protocol.dumps()[:800])

---
qodec.yaml:
  name: C4
  description: 'Built from the ''C4'' stabilizer code ([[4, 2]]). Strategy: bare-css/v1.'
  layers:
  - instruction_set: C4.isa.yaml
    codes:
      C4: C4.code.yaml
    gadgets:
      idle: idle.gadget.yaml
      measure_x: measure_x.gadget.yaml
      measure_z: measure_z.gadget.yaml
      prepare_x: prepare_x.gadget.yaml
      prepare_z: prepare_z.gadget.yaml
      transversal_cx: transversal_cx.gadget.yaml
  - instruction_set: stim.isa.yaml
  metadata:
    qdk.ec:
      build:
        code: C4
        flags_per_stabilizer: 0
        logical_qubits: 2
        omitted:
          transversal_h:
            kind: ActionMismatch
            message: logical action differs between declared and realized
            stage: verification
        physical_qubits: 4
    


### Inspect the idle gadget

Let's start with `idle`. It should leave both logical qubits unchanged while measuring the two stabilizers. Data qubits 0-3 hold the code; ancillas 4 and 5 collect the syndrome.

The gadget's YAML shows the circuit and its declarations. A _check_ is a parity equation that should evaluate to zero without faults. It can combine measurement bits with input and output encoding signs. The containing layer supplies the code and instruction-set definitions.

But declarations alone don't tell us whether the circuit does the right thing. As with `CodeProfile`, we can use a `GadgetProfile` to analyze it. Compare its `objective`, the declared operation, with its `action`, the behavior computed from the circuit.

In [4]:
gadgets["idle"]

circuit:
  source: |
    R 4
    H 4
    CX 4 0
    CX 4 1
    CX 4 2
    CX 4 3
    H 4
    R 5
    H 5
    CZ 5 0
    CZ 5 1
    CZ 5 2
    CZ 5 3
    H 5
    M 4 5
  format: stim
  in:
    '0': qubit
    '1': qubit
    '2': qubit
    '3': qubit
  out:
    '0': qubit
    '1': qubit
    '2': qubit
    '3': qubit
in:
- C4: [0, 1, 2, 3]
out:
- C4: [0, 1, 2, 3]
checks:
- ['circuit.readouts[0]', 'in[0].stabilizers[0]']
- ['circuit.readouts[1]', 'in[0].stabilizers[1]']
- ['circuit.readouts[0]', 'out[0].stabilizers[0]']
- ['circuit.readouts[1]', 'out[0].stabilizers[1]']


In [5]:
bare_idle_profile = ec.GadgetProfile(gadgets["idle"])
objective = bare_idle_profile.objective
print(bare_idle_profile.action)
assert objective is not None
assert bare_idle_profile.action.is_equivalent_to(objective)

observables: FrameGroup(generators=())
stabilizers: FrameGroup(generators=())
mapping: {X: X, Z: Z, IX: IX, IZ: IZ}


## 3. Check the qodec for correctness

We've inspected one gadget. Now we'd like to check the whole protocol. QDK provides `audit` to compare the circuits with their declared operations and equations. Our baseline should have no diagnostics; checking `diagnostics` also catches warnings that `report.ok` allows.

To see what the audit can tell us, we will deliberately remove some equations. The circuits will stay the same. Then we will use `ec.filled` to derive the missing equations and check the protocol again.

In [6]:
initial_report = ec.audit(protocol)
print(initial_report)
assert not initial_report.diagnostics

audit: ok (no diagnostics)


### Remove the measurement readouts

The `measure_x` gadget should report two logical measurement results. Its readout equations tell us which physical measurement bits and frame signs to combine for each result.

Let's clear those equations. The circuit still measures the qubits, but the protocol no longer defines its logical results. The audit should report two `gadget/missing-observable` errors and show where the equations are missing.

In [7]:
gadgets["measure_x"].readouts.clear()
print(ec.audit(protocol))

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x'] (C4 -> stim)
readouts[0] has no equation for the required logical X_0 measurement
    Expected: 2 observable bindings; declared: 0.

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x'] (C4 -> stim)
readouts[1] has no equation for the required logical X_1 measurement
    Expected: 2 observable bindings; declared: 0.

audit: 2 error(s), 0 warning(s), 0 informational


### Remove the CNOT checks

Now let's remove the four checks from `transversal_cx`. These relate the input and output stabilizer signs. The circuit still performs the logical CNOT, but we have removed the equations that specify its output stabilizer signs. The audit should add four `gadget/incomplete-output-frame` warnings to the two readout errors.

We can recover the equations with `ec.filled(protocol)`. It derives the relations from the circuits, including their dependence on incoming frame signs. It returns a new qodec, so we assign it to `protocol` and refresh `gadgets`. The audit should then be clean again.

In [8]:
gadgets["transversal_cx"].checks.clear()
print(ec.audit(protocol))

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x'] (C4 -> stim)
readouts[0] has no equation for the required logical X_0 measurement
    Expected: 2 observable bindings; declared: 0.

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x'] (C4 -> stim)
readouts[1] has no equation for the required logical X_1 measurement
    Expected: 2 observable bindings; declared: 0.

[WARNING] gadget/incomplete-output-frame
layers[0].gadgets['transversal_cx'] (C4 -> stim)
Declared relations do not determine out[0].stabilizers[0] (X_0 X_1 X_2 X_3)
    Code: C4; circuit support: ['0', '1', '2', '3'].
    Verified relation: ["out[0].stabilizers[0]", "in[0].stabilizers[0]", "in[1].stabilizers[0]"]

[WARNING] gadget/incomplete-output-frame
layers[0].gadgets['transversal_cx'] (C4 -> stim)
Declared relations do not determine out[0].stabilizers[1] (Z_0 Z_1 Z_2 Z_3)
    Code: C4; circuit support: ['0', '1', '2', '3'].
    Verified relation: ["out[0].stabilizers[1]", "in[0].stabilizers[

In [9]:
protocol = ec.filled(protocol)
gadgets = protocol.layers[0].gadgets
print(ec.audit(protocol))

audit: ok (no diagnostics)


## 4. Check the gadgets for fault tolerance

The qodec passes the audit. But what happens when a circuit has a fault? A correct noiseless circuit can still let one fault cause an undetected logical error.

We checked the code distance in section 1. Now we will use `GadgetProfile.distance()` to check the _gadget distance_: the fewest allowed circuit faults that change the logical action, stay within the output codespaces, and leave every check and flag zero.

The default fault model allows a Pauli error after a call, a flip of that call's recorded readout bits, or both. Each faulty call costs one, even if it affects several qubits or bits. A readout flip changes the recorded bit, not the surviving quantum state.

Let's compute the distance of each gadget. The search runs without a cutoff. Each result includes a _witness_: a set of faults that achieves the reported distance, together with their combined action.

In [10]:
from IPython.display import Markdown, display

def report_gadget_distances(protocol: qodec.Qodec):
    distances = {
        mnemonic: ec.GadgetProfile(gadget).distance()
        for mnemonic, gadget in protocol.layers[0].gadgets.items()
    }
    rows = [
        "| Gadget | Distance | Witness |",
        "| --- | ---: | --- |",
    ]
    rows.extend(
        f"| `{mnemonic}` | {distance} | {distance.witness} |"
        for mnemonic, distance in sorted(distances.items())
    )
    display(Markdown("\n".join(rows)))


report_gadget_distances(protocol)

| Gadget | Distance | Witness |
| --- | ---: | --- |
| `idle` | 1 | X_4 after call 3 |
| `measure_x` | 2 | X_0 after call 0; X_1 after call 1 |
| `measure_z` | 2 | flip call 0 readout 0; flip call 1 readout 0 |
| `prepare_x` | 1 | X_5 after call 18 |
| `prepare_z` | 1 | X_4 after call 7 |
| `transversal_cx` | 2 | X_4 after call 0; X_5 after call 1 |

## 5. Increase the gadget distance

### Improve the idle gadget

We would like the idle gadget to detect the faults that limit its distance. We will replace its circuit with the C4 syndrome circuit from [Reichardt, page 4, Sec. II.2](https://arxiv.org/pdf/1804.06995#page=4). Its gate order lets each syndrome ancilla detect dangerous faults on the other. It uses eight CNOTs and the same two ancillas, without adding a flag qubit.

The paper labels its data qubits 1-4; we use 0-3. Ancillas 4 and 5 measure the X and Z stabilizers. We assume these couplings are available: we are not accounting for routing swaps, movement errors, or extra idle-fault locations.

Let's compare the circuit diagrams, replace the idle circuit, and use `ec.filled` to recompute its equations. We don't need to clear the old checks first. Then we can check whether the distance improved and the audit still passes.

In [11]:
import stim

stim.Circuit(gadgets["idle"].circuit.source).diagram()

q0: -----X-----------@-------------------------
         |           |
q1: -----|-X---------|-@-----------------------
         | |         | |
q2: -----|-|-X-------|-|-@---------------------
         | | |       | | |
q3: -----|-|-|-X-----|-|-|-@-------------------
         | | | |     | | | |
q4: -R-H-@-@-@-@-H---|-|-|-|-M:rec[0]----------
                     | | | |
q5: -------------R-H-@-@-@-@-H--------M:rec[1]-

In [12]:
reichardt_source = """R 4 5
H 4
CX 4 0
CX 2 5
CX 0 5
CX 1 5
CX 4 2
CX 4 3
CX 4 1
CX 3 5
H 4
M 4 5
"""
stim.Circuit(reichardt_source).diagram()

q0: -----X---@----------------------
         |   |
q1: -----|---|-@-----X--------------
         |   | |     |
q2: -----|-@-|-|-X---|--------------
         | | | | |   |
q3: -----|-|-|-|-|-X-|-@------------
         | | | | | | | |
q4: -R-H-@-|-|-|-@-@-@-|-H-M:rec[0]-
           | | |       |
q5: -R-----X-X-X-------X---M:rec[1]-

In [13]:
gadgets["idle"].circuit.source = reichardt_source
gadgets["idle"] = ec.filled(gadgets["idle"])

idle_profile = ec.GadgetProfile(gadgets["idle"])
print("Improved idle distance:", idle_profile.distance())
print(ec.audit(protocol))

Improved idle distance: 2
audit: ok (no diagnostics)


### How the idle ancillas check each other

The distance tells us that the circuit improved. We'd also like to understand _why_. Let's follow two faults through the remaining gates.

An X error on ancilla 4 after `CX 4 2` spreads to data qubits 3 and 1. Qubit 3 still couples to ancilla 5, so the error flips the Z-syndrome result. Qubit 1 has already coupled to that ancilla and cannot cancel the flip.

Similarly, a Z error on ancilla 5 after `CX 0 5` spreads to data qubit 1 through `CX 1 5`. Then `CX 4 1` carries it to ancilla 4, where the final H turns it into an X error and flips the X-syndrome result. The Z error reaching qubit 3 arrives too late to cancel this detection.

We can check this with `FaultEvent.after` and `idle_profile.effects_of`. Each event specifies a Pauli error, a readout flip indexed within that call, or both. The profile evaluates each supplied event separately. Use `*` to combine events; applying the same Pauli fault twice cancels it. `FaultEvent()` is the identity, and `weight` counts Pauli factors and bit flips, not distance-search cost.

Each result is a `FaultEffect`: an immutable set of references to changed checks, readouts, and output signs. Its `checks`, `readouts`, and `frames` properties separate those references. Here `out[0].z[1]` means the logical-Z sign of qubit 1 in output block 0 changed, as an X error would cause. It does not mean an equation in `Gadget.frames` changed. Use `^` to combine effects for the same gadget; a target flipped twice cancels.

The references also let us inspect the declarations. `path` preserves the original text, `segments` shows its structure, and equality compares normalized addresses. `expand()` expands the final slice or union. A reference has no owner; `gadget.resolve(reference)` locates it in a gadget, and `node.value()` returns the same value as ordinary field access. It does not evaluate the equation.

Checks 0 and 2 use the X-syndrome bit; checks 1 and 3 use the Z-syndrome bit. Each pair connects one measurement to the input and output stabilizer signs, so two firing checks are not two independent detections. The final loop lists output changes without firing checks; in general, output-sign changes alone do not establish a logical failure.

In [14]:
idle_calls = gadgets["idle"].circuit.calls()
for index, call in enumerate(idle_calls):
    print(f"{index}: {call.mnemonic} {call.operands}")

0: R [4]
1: R [5]
2: H [4]
3: CX [4, 0]
4: CX [2, 5]
5: CX [0, 5]
6: CX [1, 5]
7: CX [4, 2]
8: CX [4, 3]
9: CX [4, 1]
10: CX [3, 5]
11: H [4]
12: M [4]
13: M [5]


In [15]:
x_hook = ec.FaultEvent.after(7, ec.Pauli("X_4"))
z_hook = ec.FaultEvent.after(5, ec.Pauli("Z_5"))
readout_fault = ec.FaultEvent.after(len(idle_calls) - 1, readout_flips=0)
faults = [x_hook, z_hook, readout_fault, x_hook * readout_fault]
x_effect, z_effect, readout_effect, combined_effect = idle_profile.effects_of(faults)

for fault, fault_effect in zip(faults, [x_effect, z_effect, readout_effect, combined_effect]):
    print(f"{fault} (weight {fault.weight}): {fault_effect}")

X_4 after call 7 (weight 1): ['checks[1]', 'checks[3]', 'out[0].z[1]']
Z_5 after call 5 (weight 1): ['checks[0]', 'checks[2]', 'out[0].x[0]']
flip call 13 readout 0 (weight 1): ['checks[1]', 'checks[3]']
(X_4 after call 7; flip call 13 readout 0) (weight 2): ['out[0].z[1]']


In [16]:
print("Same fault applied twice:", x_hook * x_hook)
print("XOR matches the combined event:", combined_effect == x_effect ^ readout_effect)

Same fault applied twice: no fault
XOR matches the combined event: True


In [17]:
reference = x_effect.checks[0]
print("Authored path:", reference.path)
print("Typed segments:", reference.segments)
print("Canonical reference:", reference.expand())
print("Same address as the changed output sign:", reference == x_effect.frames[0])

for path in ("checks[0:4:2]", "checks[3,1,3]"):
    selection = qodec.Reference(path)
    print(f"{path} expands to:", selection.expand())
    print("Resolved targets:", [node.path for node in gadgets["idle"].resolve(selection).sequence_nodes()])

Authored path: checks[1]
Typed segments: (Reference.Field(name='checks'), Reference.Index(value=1))
Canonical reference: [Reference('checks[1]')]
Same address as the changed output sign: False
checks[0:4:2] expands to: [Reference('checks[0]'), Reference('checks[2]')]
Resolved targets: ['checks[0]', 'checks[2]']
checks[3,1,3] expands to: [Reference('checks[3]'), Reference('checks[1]'), Reference('checks[3]')]
Resolved targets: ['checks[3]', 'checks[1]', 'checks[3]']


In [ ]:
node = gadgets["idle"].resolve(reference)
display("Check equation:", node.value(tuple))
display(gadgets["idle"].checks[1])

Resolved node: Node(path='checks[1]', type='sequence')


'Check equation:'

(circuit.readouts[1], in[0].stabilizers[1])

(circuit.readouts[1], in[0].stabilizers[1])

In [27]:
gadgets["idle"].checks[1]

(circuit.readouts[1], in[0].stabilizers[1])

In [19]:
for event, effect in idle_profile.fault_effects[:10]:
    print(f"{event} → {effect}")
print("...")

X_4 after call 0 → ['checks[0]', 'checks[2]']
Z_4 after call 0 → []
X_5 after call 1 → ['checks[1]', 'checks[3]']
Z_5 after call 1 → []
X_4 after call 2 → []
Z_4 after call 2 → ['checks[0]', 'checks[2]']
X_0 after call 3 → ['checks[1]', 'out[0].z[0]', 'out[0].z[1]', 'out[0].stabilizers[1]']
Z_0 after call 3 → ['checks[2]', 'out[0].x[0]', 'out[0].x[1]', 'out[0].stabilizers[0]']
X_4 after call 3 → ['checks[1]', 'out[0].z[0]', 'out[0].z[1]', 'out[0].stabilizers[1]']
Z_4 after call 3 → ['checks[0]', 'checks[2]']
...


In [20]:
undetected_count = 0
for event, effect in idle_profile.fault_effects:
    syndrome_length = len(effect.checks)
    error_weight = len(effect.readouts) + len(effect.frames)
    if error_weight > 0 and syndrome_length == 0:
        print(f"{event} → {effect}")
        undetected_count += 1
print(f"{undetected_count} undetected logical errors")

0 undetected logical errors


### Improve both preparations with a flag

We also need to improve the preparation gadgets. For logical $|00\rangle$, we can reset the four data qubits, apply H to qubit 0, and use three CNOTs to prepare $(|0000\rangle + |1111\rangle)/\sqrt{2}$. Adding H gates on all four qubits gives logical $|++\rangle$ in our C4 basis.

But a fault during preparation can spread to an undetected logical error. For example, an X error on qubit 0 after `CX 0 2` spreads to `X_0 X_3`. It leaves the state in the codespace but changes its logical value. The final H gates in `prepare_x` turn the same error into `Z_0 Z_3`.

We will add a _flag qubit_ to detect this spread. Reset qubit 4, place a `CX 0 4` before and after the three data CNOTs, and measure it as `reject`. The two added gates cancel without faults, but the dangerous X error flips the flag. Hiding it requires a second fault. Each preparation now uses five CNOTs and one flag qubit, without a separate syndrome-extraction circuit.

Let's declare `reject`, bind it to the measurement, and use `ec.filled` to derive the remaining equations. Then check both distances. The gadget reports the flag; the caller decides whether to discard the result.

In [21]:
flagged_preparation_source = """R 0 1 2 3 4
H 0
CX 0 4
CX 0 1
CX 0 2
CX 0 3
CX 0 4
"""
preparation_suffixes = {
    "prepare_z": "M 4\n",
    "prepare_x": "H 0 1 2 3\nM 4\n",
}
for mnemonic, suffix in preparation_suffixes.items():
    gadget = gadgets[mnemonic]
    gadget.implements.flags = ["reject"]
    gadget.circuit.source = flagged_preparation_source + suffix
    gadget.readouts = [{"reject": ["circuit.readouts[0]"]}]
    gadgets[mnemonic] = ec.filled(gadget)
    print(mnemonic, "distance:", ec.GadgetProfile(gadgets[mnemonic]).distance())
print(ec.audit(protocol))

prepare_z distance: 2
prepare_x distance: 2


audit: ok (no diagnostics)


## 6. Check the improved gadgets

We've changed the idle and preparation circuits. Now let's check the whole qodec again. The audit should have no diagnostics, and all six gadgets should have distance two under our fault model. Both preparations should also report their `reject` flag.

We need both checks: the audit tests noiseless correctness, while the distance search tests resistance to faults.

In [22]:
protocol.description = (
    "C4 built with bare-css/v1, with self-checking idle "
    "and single-flag preparation circuits."
)
print(ec.audit(protocol))
report_gadget_distances(protocol);

audit: ok (no diagnostics)


| Gadget | Distance | Witness |
| --- | ---: | --- |
| `idle` | 2 | X_4 after call 0; Z_5 after call 5 |
| `measure_x` | 2 | X_0 after call 0; X_1 after call 1 |
| `measure_z` | 2 | flip call 0 readout 0; flip call 1 readout 0 |
| `prepare_x` | 2 | X_1 after call 1; X_2 after call 2 |
| `prepare_z` | 2 | X_1 after call 1; X_2 after call 2 |
| `transversal_cx` | 2 | X_4 after call 0; X_5 after call 1 |

### Save the qodec

We now have a qodec we'd like to keep. Use `save(directory, single_file=True)` to write its circuits and declarations into one bundle. The method returns the saved path, which we can pass to `Qodec.load`.

We'll use a temporary directory here and check that loading the bundle gives us the same protocol.

In [23]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    saved_path = protocol.save(directory, single_file=True)
    assert qodec.Qodec.load(saved_path) == protocol
print("Round trip: OK")

Round trip: OK


## 7. Evaluate performance (TODO)

## What we established

We started with the C4 code and built a baseline qodec. We used the audit to find missing equations, then used gadget profiles to find faults and test better circuits. The resulting six gadgets pass the audit and each has distance two. We can also save and load the protocol without changing its declarations.

These results apply to individual gadgets under our Pauli and readout-fault model. To assess a complete implementation, we would still need to account for hardware connectivity, timing, additional fault locations, correlated noise, and how the gadgets work together.